In [10]:
%%R
# Install and load Psychometrics packages
install.packages(c("eRm", "ltm", "psych", "quantreg", "effsize", "dplyr"), repos = "http://cran.us.r-project.org")
library(eRm)        # Rasch
library(ltm)        # 2PL IRT
library(psych)      # Descriptive stats
library(quantreg)   # Quantile regression
library(effsize)    # Effect size
library(dplyr)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘mvtnorm’, ‘expm’, ‘admisc’, ‘colorspace’, ‘msm’, ‘polycor’, ‘mnormt’, ‘GPArotation’, ‘SparseM’, ‘MatrixModels’

trying URL 'http://cran.us.r-project.org/src/contrib/mvtnorm_1.3-3.tar.gz'
trying URL 'http://cran.us.r-project.org/src/contrib/expm_1.0-0.tar.gz'
trying URL 'http://cran.us.r-project.org/src/contrib/admisc_0.39.tar.gz'
trying URL 'http://cran.us.r-project.org/src/contrib/colorspace_2.1-2.tar.gz'
trying URL 'http://cran.us.r-project.org/src/contrib/msm_1.8.2.tar.gz'
trying URL 'http://cran.us.r-project.org/src/contrib/polycor_0.8-2.tar.gz'
trying URL 'http://cran.us.r-project.org/src/contrib/mnormt_2.1.1.tar.gz'
trying URL 'http://cran.us.r-project.org/src/contrib/GPArotation_2025.3-1.tar.gz'
trying URL 'http://cran.us.r-project.org/src/contrib/SparseM_1.84-2.tar.gz'
trying URL 'http://cran.us.r-project.org/src/contrib/MatrixModels_0.5-4.tar.gz'
trying URL 'htt

In [3]:
%load_ext rpy2.ipython

In [5]:
%%R
data <- data.frame(
  item_id = 1:300,
  exam_system = factor(rep(c("IB", "AP", "ALevel"), each = 100)),
  bloom = sample(0:3, 300, replace = TRUE),          # 0–3
  representation = sample(0:2, 300, replace = TRUE), # 0–2
  openness = sample(0:2, 300, replace = TRUE),       # 0–2
  modeling = sample(0:2, 300, replace = TRUE),       # 0–2
  CDI = sample(0:9, 300, replace = TRUE),            # 0–9
  response = sample(0:1, 300, replace = TRUE)        # Assuming 0/1 for response for now
)

In [11]:
%%R
describeBy(data$CDI, data$exam_system)


 Descriptive statistics by group 
group: ALevel
   vars   n mean   sd median trimmed  mad min max range skew kurtosis   se
X1    1 100 4.56 2.84      5     4.6 2.97   0   9     9 -0.2    -1.14 0.28
------------------------------------------------------------ 
group: AP
   vars   n mean   sd median trimmed  mad min max range  skew kurtosis   se
X1    1 100 4.73 2.91      5    4.72 4.45   0   9     9 -0.01    -1.38 0.29
------------------------------------------------------------ 
group: IB
   vars   n mean   sd median trimmed  mad min max range  skew kurtosis   se
X1    1 100 4.97 2.81      5    5.01 2.97   0   9     9 -0.12    -1.26 0.28


In [14]:
%%R
# Simulate additional item response columns since RM requires multiple items.
# Assuming a total of 5 items for demonstration purposes.
# In a real scenario, these would be actual item responses from your data.
data$item_resp_1 <- sample(0:1, 300, replace = TRUE)
data$item_resp_2 <- sample(0:1, 300, replace = TRUE)
data$item_resp_3 <- sample(0:1, 300, replace = TRUE)
data$item_resp_4 <- sample(0:1, 300, replace = TRUE)
data$item_resp_5 <- sample(0:1, 300, replace = TRUE)

# Combine the original 'response' column with the new dummy item response columns
# to create the item response matrix for the Rasch model.
# Assuming 'data$response' was intended as 'item_resp_0' or similar.
response_matrix_for_rasch <- as.matrix(data[, c("response", "item_resp_1", "item_resp_2", "item_resp_3", "item_resp_4", "item_resp_5")])

rasch_model <- RM(response_matrix_for_rasch)
summary(rasch_model)


Results of RM estimation: 

Call:  RM(X = response_matrix_for_rasch) 

Conditional log-likelihood: -767.2776 
Number of iterations: 3 
Number of parameters: 5 

Item (Category) Difficulty Parameters (eta): with 0.95 CI:
            Estimate Std. Error lower CI upper CI
item_resp_1    0.080      0.105   -0.127    0.286
item_resp_2   -0.013      0.105   -0.219    0.193
item_resp_3    0.053      0.105   -0.153    0.259
item_resp_4   -0.119      0.105   -0.325    0.087
item_resp_5    0.013      0.105   -0.193    0.219

Item Easiness Parameters (beta) with 0.95 CI:
                 Estimate Std. Error lower CI upper CI
beta response       0.013      0.105   -0.193    0.219
beta item_resp_1   -0.080      0.105   -0.286    0.127
beta item_resp_2    0.013      0.105   -0.193    0.219
beta item_resp_3   -0.053      0.105   -0.259    0.153
beta item_resp_4    0.119      0.105   -0.087    0.325
beta item_resp_5   -0.013      0.105   -0.219    0.193



In [20]:
%%R
library(eRm)

# Infit / Outfit statistics
fit_stats <- person.parameter(rasch_model)
summary(fit_stats)


Estimation of Ability Parameters

Collapsed log-likelihood: -17.16237 
Number of iterations: 6 
Number of parameters: 5 

ML estimated ability parameters (without spline interpolated values): 
                Estimate Std. Err.      2.5 %    97.5 %
theta P1   -1.6107687443 1.0957202 -3.7583410 0.5368035
theta P2   -0.6938008333 0.8663992 -2.3919121 1.0043104
theta P3    1.6107574555 1.0957972 -0.5369656 3.7584805
theta P4   -0.6938008333 0.8663992 -2.3919121 1.0043104
theta P5   -1.6107687443 1.0957202 -3.7583410 0.5368035
theta P6    0.0000140074 0.8169023 -1.6010850 1.6011131
theta P7    0.0000140074 0.8169023 -1.6010850 1.6011131
theta P8    1.6107574555 1.0957972 -0.5369656 3.7584805
theta P9   -1.6107687443 1.0957202 -3.7583410 0.5368035
theta P10   0.6938191898 0.8664168 -1.0043265 2.3919648
theta P11  -0.6938008333 0.8663992 -2.3919121 1.0043104
theta P12   0.0000140074 0.8169023 -1.6010850 1.6011131
theta P13   0.0000140074 0.8169023 -1.6010850 1.6011131
theta P14   0.00001400

In [23]:
%%R
irt_2pl <- ltm(response_matrix_for_rasch ~ z1, IRT.param = TRUE)
summary(irt_2pl)


Call:
ltm(formula = response_matrix_for_rasch ~ z1, IRT.param = TRUE)

Model Summary:
   log.Lik      AIC      BIC
 -1238.076 2500.152 2544.598

Coefficients:
                     value std.err  z.vals
Dffclt.response     0.0073  0.0762  0.0953
Dffclt.item_resp_1 -0.8822  1.5212 -0.5799
Dffclt.item_resp_2 -0.0315  0.2735 -0.1152
Dffclt.item_resp_3 -0.5063  0.9841 -0.5145
Dffclt.item_resp_4 -0.2414  0.3302 -0.7311
Dffclt.item_resp_5  0.2008  0.6158  0.3261
Dscrmn.response     4.2130 17.6820  0.2383
Dscrmn.item_resp_1 -0.1214  0.1642 -0.7397
Dscrmn.item_resp_2 -0.4428  0.2839 -1.5597
Dscrmn.item_resp_3 -0.1590  0.2098 -0.7579
Dscrmn.item_resp_4  0.4019  0.2553  1.5745
Dscrmn.item_resp_5  0.2014  0.2122  0.9490

Integration:
method: Gauss-Hermite
quadrature points: 21 

Optimization:
Convergence: 0 
max(|grad|): 0.0041 
quasi-Newton: BFGS 



In [28]:
%%R
params <- coef(irt_2pl)
# print(colnames(params)) # No longer needed after identifying column names
data$difficulty <- params[, "Dffclt"]
data$discrimination <- params[, "Dscrmn"]

In [30]:
%%R
qr_model <- rq(difficulty ~ CDI + exam_system, tau = 0.5, data = data)
summary(qr_model)


Call: rq(formula = difficulty ~ CDI + exam_system, tau = 0.5, data = data)

tau: [1] 0.5

Coefficients:
              coefficients lower bd upper bd
(Intercept)   -0.24143     -0.34810  0.35431
CDI            0.02332     -0.07221  0.04083
exam_systemAP  0.00000     -0.24750  0.29395
exam_systemIB  0.00000     -0.48441  0.30736


In addition: Warning messages:
1: In rq.fit.br(x, y, tau = tau, ...) : Solution may be nonunique
2: In rq.fit.br(x, y, tau = tau, ci = TRUE, ...) :
  Solution may be nonunique


In [32]:
%%R
wilcox.test(CDI ~ exam_system, data = subset(data, exam_system %in% c("IB","AP")))


	Wilcoxon rank sum test with continuity correction

data:  CDI by exam_system
W = 4763, p-value = 0.561
alternative hypothesis: true location shift is not equal to 0



In [37]:
%%R
library(effsize)

group_ib_cdi <- subset(data, exam_system == "IB")$CDI
group_ap_cdi <- subset(data, exam_system == "AP")$CDI

cliff.delta(group_ib_cdi, group_ap_cdi)


Cliff's Delta

delta estimate: 0.0474 (negligible)
95 percent confidence interval:
     lower      upper 
-0.1122803  0.2046939 



In [36]:
%%R
# Check standard deviation of CDI for IB and AP groups
cdi_ib_sd <- sd(subset(data, exam_system == "IB")$CDI)
cdi_ap_sd <- sd(subset(data, exam_system == "AP")$CDI)

cat("Standard Deviation of CDI for IB group:", cdi_ib_sd, "\n")
cat("Standard Deviation of CDI for AP group:", cdi_ap_sd, "\n")

# If standard deviation is 0 for either group, regenerate data
if (cdi_ib_sd == 0 || cdi_ap_sd == 0) {
  cat("Lack of variability detected in one or both groups. Regenerating data...\n")
  # Re-run the data generation cell (oKDm3T7eU_dS) by explicitly calling its content.
  # This is a workaround since directly re-executing a cell by id is not a direct command here.
  # In a real interactive session, you'd just re-run cell oKDm3T7eU_dS.
  # For demonstration, I'll include the data generation code again if variability is zero.
  # However, for a user asking 'how to fix it', just re-running is implied if this check reveals the issue.
  # Let's focus on the check first.
}

Standard Deviation of CDI for IB group: 2.812149 
Standard Deviation of CDI for AP group: 2.912373 


In [6]:
%%R
table(data$exam_system)


ALevel     AP     IB 
   100    100    100 
